In [11]:
"""
Threshold analysis for a pre-specified clinical operating point
Classify as high-risk if P(y=1) >= threshold ("at or above").

Threshold definition
--------------------------------------
Among true positives in the calibration cohort, sort calibrated probabilities
and set the threshold to the smallest probability among the
ceil(0.95 * n_positives) highest-scoring malignancies.

Equivalently: accept at most ~5% false negatives among malignancies on this
cohort (rule-out trade-off). This is the *highest* cutoff still compatible
with the sensitivity target.
"""

import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)


def resolve_paths():
    """Allow running with cwd = testSet/ or repo root."""
    cwd = Path.cwd().resolve()
    if (cwd / "Dataset" / "cleaned_external.xlsx").exists():
        testset_dir = cwd
    elif (cwd / "testSet" / "Dataset" / "cleaned_external.xlsx").exists():
        testset_dir = cwd / "testSet"
    else:
        raise FileNotFoundError(
            "Cannot find Dataset/cleaned_external.xlsx. "
            "Run this notebook with cwd = testSet/ (or the repo root)."
        )
    return testset_dir, testset_dir.parent


TESTSET_DIR, REPO_ROOT = resolve_paths()
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "CenterBologna"))


def FoundFeatures(path):
    return set(pd.read_excel(path).columns)


def extract_PATIENT_INFO(db):
    ids = db.pop("ID")
    morf_codificata = db.pop("morf_codificata")
    luogo_tc = db.pop("luogoTC_codificato")
    return db, ids, morf_codificata, luogo_tc


def print_binary_distribution(y, name):
    print(f"{name} size = {len(y)}")
    print(pd.Series(y).value_counts(dropna=False))


def wilson_ci(n_success, n_total, z=1.96):
    """
    Wilson score 95% CI for a binomial proportion.

    Same formula as in testCalibratedModel.ipynb
    """
    if n_total == 0:
        return (np.nan, np.nan)
    p = n_success / n_total
    q = 1.0 - p
    termine_base = 2 * n_total * p + z**2
    radice = z * np.sqrt(z**2 + 4 * n_total * p * q)
    denominatore_completo = 2 * (n_total + z**2)
    limite_inf = (termine_base - radice) / denominatore_completo
    limite_sup = (termine_base + radice) / denominatore_completo
    return float(limite_inf), float(limite_sup)


def find_threshold_for_sensitivity(y_true, y_prob, target_sensitivity=0.95):
    """
    Lock the highest cutoff compatible with a pre-specified sensitivity.

    Clinical reading
    ----------------
    - You are taking (approx.) the 5th percentile / lower-tail cutoff among
      malignancies: accept ~5% missed cancers on this locking cohort.
    """
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob, dtype=float)

    # Calibrated probs of malignancies only, low -> high
    pos_scores = np.sort(y_prob[y_true == 1])
    n_pos = int(pos_scores.size)
    if n_pos == 0:
        raise ValueError("No positive samples in y_true; cannot set a sensitivity threshold.")

    # Minimum #TP required to meet the sensitivity target on this cohort
    n_needed = int(np.ceil(target_sensitivity * n_pos))
    max_fn_allowed = n_pos - n_needed  #  (~5% of malignancies)

    # Highest threshold that still classifies the n_needed top positives as +
    # (with >=): the smallest score among those n_needed positives.
    threshold = float(pos_scores[n_pos - n_needed])

    y_pred = (y_prob >= threshold).astype(int)
    achieved = float(recall_score(y_true, y_pred, zero_division=0))
    if achieved < target_sensitivity:
        raise RuntimeError(
            f"Locked threshold {threshold} achieved sensitivity {achieved} "
            f"< target {target_sensitivity}."
        )

    # Empirical percentile rank of the threshold among positive scores (for Methods)
    # Fraction of malignancies with score < threshold (= FN rate among positives
    # if there are no ties at the threshold).
    empirical_fn_frac = float(np.mean(pos_scores < threshold))

    return threshold, achieved, n_needed, max_fn_allowed, empirical_fn_frac


In [12]:
# Calibration = non-Bologna centres (luogoTC_codificato != 1)
or_db = pd.read_excel(REPO_ROOT / "CenterBologna" / "Dataset" / "Cleaned_dataset.xlsx")
calib_db = or_db[or_db["luogoTC_codificato"] != 1].copy()

y_calibration = calib_db.pop("maligno")
X_cal_model, ids_cal, morf_cal, luogo_tc_cal = extract_PATIENT_INFO(calib_db)

print("Calibration (other centers):", X_cal_model.shape)
print_binary_distribution(y_calibration, "y_calibration")

# External validation cohort (primary performance claims)
X_test = pd.read_excel(TESTSET_DIR / "Dataset" / "cleaned_external.xlsx")
X_test, ids_test, morf_test, luogo_tc_test = extract_PATIENT_INFO(X_test)
y_test = X_test.pop("maligno")

print("Test (external):", X_test.shape)
print_binary_distribution(y_test, "y_test")

if list(X_cal_model.columns) != list(X_test.columns):
    raise ValueError("Feature columns differ between calibration and test sets.")


Calibration (other centers): (509, 124)
y_calibration size = 509
maligno
0    415
1     94
Name: count, dtype: int64
Test (external): (334, 124)
y_test size = 334
maligno
0    234
1    100
Name: count, dtype: int64


In [13]:
MODEL_PATH = (
    REPO_ROOT / "CenterBologna" / "Elements" / "Boruta_RusBoost" / "deploy_modelCalibrated_model.pkl"
)
FEATURES_PATH = (
    REPO_ROOT
    / "CenterBologna"
    / "Elements"
    / "Boruta_RusBoost"
    / "deploy_modelBoruta_RusBoost_ReduceDataset.xlsx"
)

calibrated_pipeline = joblib.load(MODEL_PATH)
n_features = len(FoundFeatures(FEATURES_PATH))

print("Loaded:", MODEL_PATH)
print("#Features:", n_features)
print("classes_:", calibrated_pipeline.classes_)
print("Calibration method:", calibrated_pipeline.method)

if list(calibrated_pipeline.classes_) != [0, 1]:
    raise ValueError(
        f"Unexpected classes_={calibrated_pipeline.classes_}; "
        "code assumes column 1 of predict_proba is P(maligno=1)."
    )


Loaded: /home/simone/Scrivania/AdreCorPredictions/CenterBologna/Elements/Boruta_RusBoost/deploy_modelCalibrated_model.pkl
#Features: 42
classes_: [0 1]
Calibration method: sigmoid


In [14]:
# Pre-specified operating point: sensitivity 95% (rule-out / limited FN among malignancies).
TARGET_SENSITIVITY = 0.95

cal_probs = calibrated_pipeline.predict_proba(X_cal_model)[:, 1]

pd.DataFrame(
    {"ID": ids_cal.to_numpy(), "calibrated_prob": cal_probs}
).to_excel(TESTSET_DIR / "scores_calibrationSet.xlsx", index=False)
print("Saved:", TESTSET_DIR / "scores_calibrationSet.xlsx")

(
    threshold,
    sens_on_cal,
    n_needed,
    max_fn_allowed,
    empirical_fn_frac,
) = find_threshold_for_sensitivity(
    y_calibration, cal_probs, target_sensitivity=TARGET_SENSITIVITY
)

# High-risk if calibrated probability is at or above the locked threshold
cal_pred = (cal_probs >= threshold).astype(int)
tn_c, fp_c, fn_c, tp_c = confusion_matrix(y_calibration, cal_pred).ravel()
spec_c = tn_c / (tn_c + fp_c) if (tn_c + fp_c) else np.nan
n_pos_cal = int((np.asarray(y_calibration) == 1).sum())

print(
    f"Target sensitivity={TARGET_SENSITIVITY} on n_pos={n_pos_cal} "
    f"-> require TP>={n_needed} (allow FN<={max_fn_allowed})"
)
print(
    f"Locked threshold = {threshold:.6f}  "
    f"(~ lower-tail among malignancies; empirical P(score < thr | y=1)={empirical_fn_frac:.3f})"
)
print(f"Achieved sensitivity (calibration): {sens_on_cal:.3f}  [TP={tp_c}, FN={fn_c}]")
print(f"Specificity (calibration): {spec_c:.3f}  [TN={tn_c}, FP={fp_c}]")
print(
    "Note: calibration metrics are for threshold locking only; "
    "primary estimates are on the external test set."
)


Saved: /home/simone/Scrivania/AdreCorPredictions/testSet/scores_calibrationSet.xlsx
Target sensitivity=0.95 on n_pos=94 -> require TP>=90 (allow FN<=4)
Locked threshold = 0.157287  (~ lower-tail among malignancies; empirical P(score < thr | y=1)=0.043)
Achieved sensitivity (calibration): 0.957  [TP=90, FN=4]
Specificity (calibration): 0.904  [TN=375, FP=40]
Note: calibration metrics are for threshold locking only; primary estimates are on the external test set.


In [15]:
test_probs = calibrated_pipeline.predict_proba(X_test)[:, 1]

pd.DataFrame(
    {"ID": ids_test.to_numpy(), "calibrated_prob": test_probs}
).to_excel(TESTSET_DIR / "scores_testSet.xlsx", index=False)
print("Saved:", TESTSET_DIR / "scores_testSet.xlsx")

test_pred = (test_probs >= threshold).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test, test_pred).ravel()
n_pos_test = tp + fn
n_neg_test = tn + fp

Accuracy = round(accuracy_score(y_test, test_pred), 3)
precision = round(precision_score(y_test, test_pred, zero_division=0), 3)
recall = round(recall_score(y_test, test_pred, zero_division=0), 3)  # sensitivity
f1 = round(f1_score(y_test, test_pred, zero_division=0), 3)
auc_roc = round(roc_auc_score(y_test, test_probs), 3)  # threshold-independent
specificity = round(tn / n_neg_test, 3) if n_neg_test else 0.0

# 95% Wilson CIs for key binomial proportions (external validation)
sens_lo, sens_hi = wilson_ci(tp, n_pos_test)
spec_lo, spec_hi = wilson_ci(tn, n_neg_test)
ppv_lo, ppv_hi = wilson_ci(tp, tp + fp) if (tp + fp) else (np.nan, np.nan)

scores = pd.DataFrame(
    [[f1, recall, precision, Accuracy, auc_roc, int(n_features)]],
    columns=["F1-score", "Recall", "Precision", "Accuracy", "Auc-Score", "#Features"],
)

print(f"Frozen threshold applied on test: {threshold:.6f}")
print(f"Test CM — TN={tn}, FP={fp}, FN={fn}, TP={tp}")
print(
    f"Sensitivity {recall:.3f} (95% CI {sens_lo:.3f}-{sens_hi:.3f}) | "
    f"Specificity {specificity:.3f} (95% CI {spec_lo:.3f}-{spec_hi:.3f})"
)
print(
    f"PPV (Precision) {precision:.3f} (95% CI {ppv_lo:.3f}-{ppv_hi:.3f}) | "
    f"AUC {auc_roc:.3f}"
)
scores


Saved: /home/simone/Scrivania/AdreCorPredictions/testSet/scores_testSet.xlsx
Frozen threshold applied on test: 0.157287
Test CM — TN=195, FP=39, FN=6, TP=94
Sensitivity 0.940 (95% CI 0.875-0.972) | Specificity 0.833 (95% CI 0.780-0.876)
PPV (Precision) 0.707 (95% CI 0.624-0.777) | AUC 0.969


,F1-score,Recall,Precision,Accuracy,Auc-Score,#Features
0,0.807,0.94,0.707,0.865,0.969,42
